In [1]:
from huggingface_hub import login
login()

In [2]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 85.1 MB/s eta 0:00:00:00:0100:01


In [5]:
!pip install -U langchain langchain-community langchain-core langchain-text-splitters pypdf sentence-transformers transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 20.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 98.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.2/250.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 94.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.3/162.3 kB 12.8 MB/s eta 0:00:00
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempti

In [7]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

/tmp/ipykernel_58/2439849371.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [9]:
model_name = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype = torch.float16,
    device_map = "auto"
)

[transformers] The tokenizer you are loading from 'mistralai/Mistral-Nemo-Instruct-2407' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [10]:
def GenerateText(prompt, max_length = 512, num_return_sequences = 1):
    inputs = tokenizer(prompt, return_tensors = "pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_length = max_length,
        num_return_sequences = num_return_sequences,
        do_sample = True,
        top_k = 50,
        top_p = 0.95,
        temperature = 0.7
    )
    answers = []
    for output in outputs:
        answer = tokenizer.decode(output, skip_special_tokens = True)
        answers.append(answer)

    return answers

In [14]:
pdf_path = "/kaggle/input/datasets/omar1234321/hindawi-university-pdf/Tips Hindawi University Info.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

text_splitter = CharacterTextSplitter(chunk_size = 1000, chunk_overlap = 100)
chunks = text_splitter.split_documents(documents)

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding = HuggingFaceEmbeddings(model_name = embedding_model_name)

vectordb = FAISS.from_documents(chunks, embedding)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [17]:
def ask_question(query):
    docs = vectordb.similarity_search(query, k = 3)
    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""You are a helpful assistant. Use the following context to answer the question.

Context:
{context}

Question: {query}
Answer:"""
    result = GenerateText(prompt, max_length = 1000)[0]
    return result.strip()

In [18]:
if __name__ == "__main__":
    print("PDF Q&A System with Mistral \n")
    while True:
        user_query = input("Ask a question (or type 'exit'): ")
        if user_query.lower() == "exit":
            break
        answer = ask_question(user_query)
        print("\n Answer:", answer, "\n")

PDF Q&A System with Mistral 



Ask a question (or type 'exit'):  When was andrew born?



 Answer: You are a helpful assistant. Use the following context to answer the question.

Context:
9. Alumni and Impact
Lina Darwish: UN Youth Ambassador
Hassan Joudeh: CEO of ArabTech
Noura Saleh: Award-winning novelist
More  than  60  alumni  chapters  exist  globally.  The  THU  Alumni  Network  hosts  quarterly  webinars,
mentorship programs, and reunions.
End of document
• 
• 
• 
4

3.2 Sample Undergraduate Programs
BSc in Computer Science
BA in International Relations
BBA in Marketing
BEng in Civil Engineering
BSc in Nursing
3.3 Graduate Programs
MBA (Master of Business Administration)
MSc in Data Science
MA in Psychology
PhD in Mechanical Engineering
PhD in Political Science
4. Admissions and Tuition
4.1 Undergraduate Admissions
High school GPA of 85% or equivalent
English proficiency test (IELTS 6.0 or TOEFL 80)
Entrance interview for certain programs
4.2 Graduate Admissions
Relevant bachelor’s degree
Minimum GPA of 3.0/4.0
Two academic references
Statement of purpose
4.3 Tuiti

Ask a question (or type 'exit'):  When was Tips Hindawi University found?



 Answer: You are a helpful assistant. Use the following context to answer the question.

Context:
1. General Overview
Tips Hindawi University (THU) is a premier institution of higher education located in the heart of the Middle
East. Founded in 1963, the university has grown into a globally recognized center for academic excellence
and innovation. With over six decades of educational leadership, THU has produced more than 150,000
graduates who serve in diverse industries and academic circles worldwide.
The university is accredited by the International Commission for Academic Standards and the Ministry of
Higher Education. It operates under the guiding motto: "Knowledge, Integrity, Progress", and fosters an
environment of research, creativity, and public service.
2. Campus and Facilities
2.1 Main Campus
The main campus is located in the capital city and spans over 300 acres. It houses: - 12 academic buildings -
Central library with over 1 million volumes - Four residence halls - A medi

...

Ask a question (or type 'exit'):  How old Omar Ayman is?



 Answer: You are a helpful assistant. Use the following context to answer the question.

Context:
9. Alumni and Impact
Lina Darwish: UN Youth Ambassador
Hassan Joudeh: CEO of ArabTech
Noura Saleh: Award-winning novelist
More  than  60  alumni  chapters  exist  globally.  The  THU  Alumni  Network  hosts  quarterly  webinars,
mentorship programs, and reunions.
End of document
• 
• 
• 
4

Vice President of Academic Affairs: Prof. Layla Mahmoud
Dean of Students: Mr . Samer Hussein
Registrar's Office: Handles academic records, course registration, transcripts
Office of International Programs: Assists international students and partnerships
6. Research and Innovation
Center for Renewable Energy Research
Artificial Intelligence and Robotics Lab
Public Policy and Development Institute
Annual "Future Frontiers" Conference
Research budget: $12 million annually
Sample Project: - "Solar-Powered Desalination Systems for Arid Regions" led by Prof. Amina Al-Rawi
7. Student Life
7.1 Clubs and Organi

Ask a question (or type 'exit'):  When was Andrew born?



 Answer: You are a helpful assistant. Use the following context to answer the question.

Context:
9. Alumni and Impact
Lina Darwish: UN Youth Ambassador
Hassan Joudeh: CEO of ArabTech
Noura Saleh: Award-winning novelist
More  than  60  alumni  chapters  exist  globally.  The  THU  Alumni  Network  hosts  quarterly  webinars,
mentorship programs, and reunions.
End of document
• 
• 
• 
4

3.2 Sample Undergraduate Programs
BSc in Computer Science
BA in International Relations
BBA in Marketing
BEng in Civil Engineering
BSc in Nursing
3.3 Graduate Programs
MBA (Master of Business Administration)
MSc in Data Science
MA in Psychology
PhD in Mechanical Engineering
PhD in Political Science
4. Admissions and Tuition
4.1 Undergraduate Admissions
High school GPA of 85% or equivalent
English proficiency test (IELTS 6.0 or TOEFL 80)
Entrance interview for certain programs
4.2 Graduate Admissions
Relevant bachelor’s degree
Minimum GPA of 3.0/4.0
Two academic references
Statement of purpose
4.3 Tuiti

KeyboardInterrupt: Interrupted by user